# Package and Validate a Customer H2O Model

Use the customer model, feature contract, and golden files from `workshop/.env` to create a checksum manifest and prove exact-version prediction parity before Azure mutation.

> The customer artifact must be an H2O binary model created with `h2o.save_model()`, not a MOJO zip.

**Source:** Adapted from this repository's H2O reference and onboarding notebooks.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

import h2o
import numpy as np
import pandas as pd
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

def workshop_path(name: str) -> Path:
    value = Path(os.environ[name])
    return value if value.is_absolute() else WORKSHOP_ROOT / value

MODEL_PATH = workshop_path("H2O_CUSTOMER_MODEL_PATH").resolve()
INPUT_PATH = workshop_path("H2O_CUSTOMER_INPUT_PATH").resolve()
EXPECTED_PATH = workshop_path("H2O_CUSTOMER_EXPECTED_PATH").resolve()
BUNDLE_DIR = MODEL_PATH.parent
H2O_VERSION = os.environ["H2O_VERSION"]
FEATURES = [value.strip() for value in os.environ["H2O_FEATURES"].split(",") if value.strip()]
CATEGORICAL_FEATURES = [value.strip() for value in os.environ["H2O_CATEGORICAL_FEATURES"].split(",") if value.strip()]
TARGET = os.environ["H2O_TARGET"]
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
MODEL_VERSION = os.environ["H2O_MODEL_VERSION"]

for path in (MODEL_PATH, INPUT_PATH, EXPECTED_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Customer input not found: {path}")
    if path.parent != BUNDLE_DIR:
        raise ValueError("Model and golden files must be colocated in the customer bundle directory")
if h2o.__version__ != H2O_VERSION:
    raise RuntimeError(f"Expected h2o=={H2O_VERSION}, found {h2o.__version__}")
if MODEL_VERSION.lower() == "latest":
    raise ValueError("Use an immutable H2O model version, not 'latest'")
if not set(CATEGORICAL_FEATURES).issubset(FEATURES):
    raise ValueError("Categorical features must be a subset of H2O_FEATURES")

golden_input = pd.read_csv(INPUT_PATH)
golden_expected = pd.read_csv(EXPECTED_PATH)
if list(golden_input.columns) != FEATURES:
    raise ValueError(f"Golden input columns must be ordered as: {FEATURES}")
if list(golden_expected.columns) != ["predict"] or len(golden_expected) != len(golden_input):
    raise ValueError("Golden expected must contain one predict value per input row")

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = {
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "model_format": "h2o_binary",
    "h2o_version": H2O_VERSION,
    "model_file": MODEL_PATH.name,
    "target": TARGET,
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "files": {
        MODEL_PATH.name: sha256(MODEL_PATH),
        INPUT_PATH.name: sha256(INPUT_PATH),
        EXPECTED_PATH.name: sha256(EXPECTED_PATH),
    },
}
(BUNDLE_DIR / "model_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from validate_bundle import validate_bundle
summary = validate_bundle(BUNDLE_DIR, H2O_VERSION)

try:
    h2o.init(max_mem_size="2G", nthreads=-1)
    model = h2o.load_model(str(MODEL_PATH))
    frame = h2o.H2OFrame(golden_input)
    for column in CATEGORICAL_FEATURES:
        frame[column] = frame[column].asfactor()
    actual = model.predict(frame).as_data_frame()
    np.testing.assert_allclose(golden_expected["predict"], actual["predict"], rtol=1e-6, atol=1e-6)
    display(summary)
    print("Customer bundle packaging and golden parity passed.")
finally:
    if h2o.connection() is not None:
        h2o.cluster().shutdown(prompt=False)

## Expected Result

The customer folder contains a checksum manifest, and the exact H2O runtime reproduces every supplied golden prediction before Azure mutation is allowed.

Next: `02_register_model.ipynb`.